# 1. Carga y Balanceo de Datos (SMOTE)

In [3]:
import pandas as pd
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
from sklearn.preprocessing import LabelEncoder

# 1. Carga de datos
df = pd.read_csv('smart_manufacturing_data.csv')

# ==========================================
# 2. PREPARACIÓN Y BALANCEO (SMOTE)
# ==========================================
columna_objetivo = 'anomaly_flag' 

columnas_a_ignorar = [columna_objetivo, 'timestamp', 'machine_id']
columnas_a_borrar = [col for col in columnas_a_ignorar if col in df.columns]

X = df.drop(columnas_a_borrar, axis=1)
y = df[columna_objetivo]

# 👇 --- EL PARCHE MÁGICO --- 👇
# Buscamos todas las columnas que sean de texto (tipo 'object') en nuestras características
columnas_texto = X.select_dtypes(include=['object']).columns

# Convertimos las palabras a números (ej. 'Normal' se vuelve 0)
le = LabelEncoder()
for col in columnas_texto:
    X[col] = le.fit_transform(X[col].astype(str)) # astype(str) por si hay valores nulos atravesados
# 👆 ------------------------ 👆

# División en Entrenamiento (80%) y Prueba (20%)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Aplicamos SMOTE para inflar los datos
smote = SMOTE(random_state=42)
X_train_bal, y_train_bal = smote.fit_resample(X_train, y_train)

# Verificamos que se cumpla la cantidad de registros
total_registros = len(X_train_bal) + len(X_test)
print(f"Total de registros (Entrenamiento + Prueba) tras balanceo: {total_registros}")
print("Distribución de clases en entrenamiento:\n", y_train_bal.value_counts())

C:\Users\Kese\AppData\Local\Temp\ipykernel_11504\2486555244.py:22: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  columnas_texto = X.select_dtypes(include=['object']).columns


Total de registros (Entrenamiento + Prueba) tras balanceo: 165742
Distribución de clases en entrenamiento:
 anomaly_flag
0    72871
1    72871
Name: count, dtype: int64


# 2. Entrenamiento de Modelos Clásicos (Random Forest, XGBoost, SVM)

In [4]:
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report

# ==========================================
# 3. ENTRENAMIENTO DE MODELOS CLÁSICOS
# ==========================================

# --- 1. Random Forest ---
print("--- Entrenando Random Forest ---")
# n_jobs=-1 usa todos los procesadores de tu compu para que vuele
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_model.fit(X_train_bal, y_train_bal)
rf_pred = rf_model.predict(X_test)

print(f"Precisión Random Forest: {accuracy_score(y_test, rf_pred) * 100:.2f}%\n")
print("Reporte Random Forest:")
print(classification_report(y_test, rf_pred))


# --- 2. XGBoost ---
print("\n--- Entrenando XGBoost ---")
xgb_model = XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
xgb_model.fit(X_train_bal, y_train_bal)
xgb_pred = xgb_model.predict(X_test)

print(f"Precisión XGBoost: {accuracy_score(y_test, xgb_pred) * 100:.2f}%\n")


# --- 3. SVM (Support Vector Machine) ---
print("\n--- Entrenando SVM ---")
svm_model = SVC(kernel='rbf', random_state=42)
svm_model.fit(X_train_bal, y_train_bal)
svm_pred = svm_model.predict(X_test)

print(f"Precisión SVM: {accuracy_score(y_test, svm_pred) * 100:.2f}%\n")



--- Entrenando Random Forest ---
Precisión Random Forest: 100.00%

Reporte Random Forest:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     18213
           1       1.00      1.00      1.00      1787

    accuracy                           1.00     20000
   macro avg       1.00      1.00      1.00     20000
weighted avg       1.00      1.00      1.00     20000


--- Entrenando XGBoost ---


c:\Users\Kese\Desktop\AgroSentinel-Predictivo\venv\Lib\site-packages\xgboost\training.py:200: UserWarning: [07:49:30] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:794: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Precisión XGBoost: 100.00%


--- Entrenando SVM ---
Precisión SVM: 98.41%



# 3. Entrenamiento de Redes Neuronales (LSTM)

In [5]:
import numpy as np
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout

# ==========================================
# 4. ENTRENAMIENTO DE REDES NEURONALES (LSTM)
# ==========================================
print("\n--- Entrenando LSTM ---")

# Las redes LSTM: (muestras, pasos_de_tiempo, características)
X_train_lstm = np.array(X_train_bal).reshape((X_train_bal.shape[0], 1, X_train_bal.shape[1]))
X_test_lstm = np.array(X_test).reshape((X_test.shape[0], 1, X_test.shape[1]))

# Construimos la red neuronal
lstm_model = Sequential()
lstm_model.add(LSTM(64, input_shape=(1, X_train_bal.shape[1]), activation='relu'))
lstm_model.add(Dropout(0.2))
lstm_model.add(Dense(32, activation='relu'))
lstm_model.add(Dense(1, activation='sigmoid'))

# Compilamos
lstm_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Entrenamos por 5 "vueltas" (épocas) para que no sea tan demorado
lstm_model.fit(X_train_lstm, y_train_bal, epochs=5, batch_size=64, validation_data=(X_test_lstm, y_test), verbose=1)

# Evaluamos la precisión final
lstm_loss, lstm_acc = lstm_model.evaluate(X_test_lstm, y_test, verbose=0)
print(f"\nPrecisión LSTM: {lstm_acc * 100:.2f}%")


--- Entrenando LSTM ---


c:\Users\Kese\Desktop\AgroSentinel-Predictivo\venv\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/5
2278/2278 ━━━━━━━━━━━━━━━━━━━━ 21s 7ms/step - accuracy: 0.9892 - loss: 0.0337 - val_accuracy: 1.0000 - val_loss: 2.9483e-05
Epoch 2/5
2278/2278 ━━━━━━━━━━━━━━━━━━━━ 14s 6ms/step - accuracy: 0.9996 - loss: 0.0014 - val_accuracy: 0.9906 - val_loss: 0.0218
Epoch 3/5
2278/2278 ━━━━━━━━━━━━━━━━━━━━ 12s 5ms/step - accuracy: 0.9998 - loss: 6.0535e-04 - val_accuracy: 1.0000 - val_loss: 7.5644e-07
Epoch 4/5
2278/2278 ━━━━━━━━━━━━━━━━━━━━ 12s 5ms/step - accuracy: 0.9996 - loss: 0.0011 - val_accuracy: 1.0000 - val_loss: 1.2802e-05
Epoch 5/5
2278/2278 ━━━━━━━━━━━━━━━━━━━━ 11s 5ms/step - accuracy: 0.9997 - loss: 9.1287e-04 - val_accuracy: 1.0000 - val_loss: 2.4426e-06

Precisión LSTM: 100.00%


In [6]:
import joblib

# ==========================================
# 5. EXPORTACIÓN DEL MODELO PARA LA WEB
# ==========================================
# Guardamos el modelo entrenado en un archivo físico
joblib.dump(rf_model, 'modelo_rf_final.pkl')

print("¡Modelo exportado exitosamente! Busca el archivo 'modelo_rf_final.pkl' en tu carpeta del proyecto.")

¡Modelo exportado exitosamente! Busca el archivo 'modelo_rf_final.pkl' en tu carpeta del proyecto.
